# XGBoost Ensemble March Madness Predictions

5-model ensemble using 10-fold GroupKFold cross-validation:
- HistGradientBoostingClassifier
- RandomForestClassifier
- LogisticRegression
- SVC (with probability)
- XGBoost

Feature engineering on 3 datasets:
1. Regular season compact results (win pcts by location + scoring)
2. Regular season detailed results (shooting %, box score rates)
3. Massey Ordinals (multiple ranking checkpoints, men's only)

Final blend: `0.3 * ensemble + 0.7 * seed_spline`

Output: `output/{CURRENT_SEASON}/xgb_ensemble/submission_combined.csv`

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
CURRENT_SEASON = 2026
# DATA_DIR options:
#   "../data/{CURRENT_SEASON}"  — year-specific dir (men's only, no women's for 2025)
#   "../data/2026"              — cumulative through 2025 (has both M and W; use for backtesting)
DATA_DIR = f"../data/{CURRENT_SEASON}"
ENSEMBLE_WEIGHT = 0.3   # weight on ensemble vs spline (0=all spline, 1=all ensemble)
N_FOLDS = 10
# =============================================================================

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.interpolate import UnivariateSpline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import log_loss
import xgboost as xgb

sys.path.insert(0, os.path.abspath('..'))

def make_output_path(year, method, gender, filename):
    path = f"../output/{year}/{method}/{gender}"
    os.makedirs(path, exist_ok=True)
    return f"{path}/{filename}"

os.makedirs("../output", exist_ok=True)
print(f"Season: {CURRENT_SEASON}")

## Feature Engineering — Compact Results (Win %)

In [3]:
def compute_compact_features(results_df):
    """Win % by location (H/A/N) and avg score margin per team-season."""
    records = []
    for (season, team_id), grp_w in results_df.groupby(['Season','WTeamID']):
        grp_l = results_df[(results_df['Season']==season) & (results_df['LTeamID']==team_id)]
        total = len(grp_w) + len(grp_l)
        if total == 0: continue

        # Win pct by location
        for loc in ['H','A','N']:
            wins_loc = len(grp_w[grp_w['WLoc']==loc])
            losses_loc = len(grp_l[grp_l['WLoc']=={'H':'A','A':'H','N':'N'}[loc]])
            total_loc = wins_loc + losses_loc
            records.append({
                'Season': season, 'TeamID': team_id,
                f'WinPct_{loc}': wins_loc/total_loc if total_loc else 0.5,
            })

    # Merge location win pcts
    from functools import reduce
    frames = {}
    for r in records:
        k = (r['Season'], r['TeamID'])
        if k not in frames: frames[k] = {'Season':r['Season'],'TeamID':r['TeamID']}
        frames[k].update({col:val for col,val in r.items() if col not in ['Season','TeamID']})

    df = pd.DataFrame(list(frames.values()))

    # Overall win pct and score margin
    all_wins = results_df.groupby(['Season','WTeamID']).size().reset_index(name='Wins')
    all_wins.rename(columns={'WTeamID':'TeamID'}, inplace=True)
    all_losses = results_df.groupby(['Season','LTeamID']).size().reset_index(name='Losses')
    all_losses.rename(columns={'LTeamID':'TeamID'}, inplace=True)
    games = all_wins.merge(all_losses, on=['Season','TeamID'], how='outer').fillna(0)
    games['WinPct_Overall'] = games['Wins'] / (games['Wins'] + games['Losses'])

    # Score margin
    results_df['WMargin'] = results_df['WScore'] - results_df['LScore']
    win_margin = results_df.groupby(['Season','WTeamID'])['WMargin'].mean().reset_index(name='AvgWinMargin')
    win_margin.rename(columns={'WTeamID':'TeamID'}, inplace=True)
    loss_margin = results_df.groupby(['Season','LTeamID'])['WMargin'].mean().reset_index(name='AvgLossMargin')
    loss_margin.rename(columns={'LTeamID':'TeamID'}, inplace=True)

    df = df.merge(games[['Season','TeamID','WinPct_Overall','Wins','Losses']], on=['Season','TeamID'], how='left')
    df = df.merge(win_margin, on=['Season','TeamID'], how='left')
    df = df.merge(loss_margin, on=['Season','TeamID'], how='left')
    df['AvgMargin'] = df['AvgWinMargin'].fillna(0) - df['AvgLossMargin'].fillna(0)
    return df.fillna(0.5)

## Feature Engineering — Detailed Results (Box Score)

In [4]:
def compute_detailed_features(detailed_df):
    """Shooting %, assist/turnover ratio, rebounding per team-season."""
    stats = []
    for season, grp in detailed_df.groupby('Season'):
        # Winner stats
        w = grp.groupby('WTeamID').agg(
            WFGM=('WFGM','mean'), WFGA=('WFGA','mean'),
            WFGM3=('WFGM3','mean'), WFGA3=('WFGA3','mean'),
            WFTM=('WFTM','mean'), WFTA=('WFTA','mean'),
            WOR=('WOR','mean'), WDR=('WDR','mean'),
            WAst=('WAst','mean'), WTO=('WTO','mean'), WStl=('WStl','mean'),
        ).reset_index().rename(columns={'WTeamID':'TeamID'})
        w['FGPct'] = w['WFGM'] / w['WFGA'].clip(lower=1)
        w['FG3Pct'] = w['WFGM3'] / w['WFGA3'].clip(lower=1)
        w['FTPct'] = w['WFTM'] / w['WFTA'].clip(lower=1)
        w['AstTORate'] = w['WAst'] / w['WTO'].clip(lower=1)
        w['RebRate'] = w['WOR'] + w['WDR']
        w['Season'] = season
        stats.append(w[['Season','TeamID','FGPct','FG3Pct','FTPct','AstTORate','RebRate','WStl']])

    return pd.concat(stats, ignore_index=True)

## Feature Engineering — Massey Rankings

In [5]:
def compute_massey_multi(massey_df):
    """Massey ranks at 3 checkpoints: early (day 50), mid (day 90), late (day 128)."""
    checkpoints = [('Early', 15, 50), ('Mid', 80, 100), ('Late', 120, 133)]
    frames = []
    for label, day_min, day_max in checkpoints:
        sub = massey_df[(massey_df['RankingDayNum'] >= day_min) &
                        (massey_df['RankingDayNum'] <= day_max)]
        avg = sub.groupby(['Season','TeamID'])['OrdinalRank'].mean().reset_index(name=f'MasseyRank_{label}')
        # Normalize: lower rank number = better team
        avg[f'MasseyNorm_{label}'] = avg.groupby('Season')[f'MasseyRank_{label}'].transform(
            lambda x: 1 - (x - x.min()) / (x.max() - x.min() + 1e-9)
        )
        frames.append(avg)

    result = frames[0]
    for f in frames[1:]:
        result = result.merge(f, on=['Season','TeamID'], how='outer')

    # Trajectory: late rank vs early rank
    result['MasseyTrajectory'] = result['MasseyNorm_Late'] - result.get('MasseyNorm_Early', result['MasseyNorm_Late'])
    return result.fillna(0.5)

## Build Training Dataset

In [6]:
def build_matchup_df(tourney_df, compact_feats, detailed_feats, seeds_df, massey_feats=None):
    """Build feature matrix with Team1/Team2 difference features."""
    all_feats = compact_feats.copy()
    if detailed_feats is not None:
        all_feats = all_feats.merge(detailed_feats, on=['Season','TeamID'], how='left')
    if massey_feats is not None:
        all_feats = all_feats.merge(massey_feats, on=['Season','TeamID'], how='left')

    feat_cols = [c for c in all_feats.columns if c not in ['Season','TeamID']]

    rows = []
    for _, g in tourney_df.iterrows():
        s, w, l = g['Season'], g['WTeamID'], g['LTeamID']
        t1, t2 = min(w,l), max(w,l)
        actual = 1.0 if t1 == w else 0.0

        f1 = all_feats[(all_feats['Season']==s) & (all_feats['TeamID']==t1)]
        f2 = all_feats[(all_feats['Season']==s) & (all_feats['TeamID']==t2)]

        if len(f1)==0 or len(f2)==0: continue
        f1, f2 = f1.iloc[0], f2.iloc[0]

        def gs(tid):
            r = seeds_df[(seeds_df['Season']==s)&(seeds_df['TeamID']==tid)]
            return int(r.iloc[0]['Seed'][1:3]) if len(r) else 8

        row = {'Season': s, 'Team1ID': t1, 'Team2ID': t2, 'Result': actual,
               'SeedDiff': gs(t1) - gs(t2)}

        for col in feat_cols:
            row[f'{col}_1'] = f1.get(col, 0)
            row[f'{col}_2'] = f2.get(col, 0)
            row[f'{col}_Diff'] = f1.get(col, 0) - f2.get(col, 0)

        rows.append(row)

    return pd.DataFrame(rows)

print("Feature building functions ready.")

Feature building functions ready.


## Men's 5-Model Ensemble

In [7]:
# Load data
reg_m = pd.read_csv(f"{DATA_DIR}/MRegularSeasonCompactResults.csv")
det_m = pd.read_csv(f"{DATA_DIR}/MRegularSeasonDetailedResults.csv")
tourney_m = pd.read_csv(f"{DATA_DIR}/MNCAATourneyCompactResults.csv")
seeds_m = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeeds.csv")
massey_m = pd.read_csv(f"{DATA_DIR}/MMasseyOrdinals.csv")
teams_m = pd.read_csv(f"{DATA_DIR}/MTeams.csv")

# Compute features
compact_m = compute_compact_features(reg_m)
detailed_m = compute_detailed_features(det_m)
massey_multi_m = compute_massey_multi(massey_m)

train_tourney_m = tourney_m[(tourney_m['Season'] >= 2010) & (tourney_m['Season'] < CURRENT_SEASON)]
feat_df_m = build_matchup_df(train_tourney_m, compact_m, detailed_m, seeds_m, massey_multi_m)

model_feat_cols = [c for c in feat_df_m.columns
                   if c not in ['Season','Team1ID','Team2ID','Result']]
X_m = feat_df_m[model_feat_cols].fillna(0)
y_m = feat_df_m['Result']
groups_m = feat_df_m['Season']

print(f"Training samples: {len(X_m)}, features: {len(model_feat_cols)}")

Training samples: 934, features: 67


In [8]:
# 5-model ensemble with 10-fold GroupKFold
models = [
    ('HistGBM', HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=42)),
    ('RF', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
    ('LR', Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=0.1, max_iter=1000))])),
    ('SVC', Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, C=1.0))])),
    ('XGB', xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               use_label_encoder=False, eval_metric='logloss',
                               random_state=42, verbosity=0)),
]

gkf = GroupKFold(n_splits=N_FOLDS)
oof_preds_m = np.zeros((len(X_m), len(models)))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_m, y_m, groups_m)):
    X_tr, X_val = X_m.iloc[train_idx], X_m.iloc[val_idx]
    y_tr, y_val = y_m.iloc[train_idx], y_m.iloc[val_idx]

    for mi, (name, clf) in enumerate(models):
        clf.fit(X_tr, y_tr)
        oof_preds_m[val_idx, mi] = clf.predict_proba(X_val)[:, 1]

    if fold % 2 == 0:
        ensemble_oof = oof_preds_m[val_idx].mean(axis=1)
        print(f"Fold {fold+1}/{N_FOLDS}: logloss = {log_loss(y_val, ensemble_oof):.4f}")

ensemble_oof_m = oof_preds_m.mean(axis=1)
print(f"Overall OOF log loss: {log_loss(y_m, ensemble_oof_m):.4f}")

Fold 1/10: logloss = 0.5766
Fold 3/10: logloss = 0.6484
Fold 5/10: logloss = 0.6372
Fold 7/10: logloss = 0.5574
Fold 9/10: logloss = 0.6004
Overall OOF log loss: 0.5927


In [9]:
# Retrain all models on full dataset for prediction
trained_models_m = []
for name, clf in models:
    clf.fit(X_m, y_m)
    trained_models_m.append((name, clf))

# Seed spline for blend
seed_diff_m = feat_df_m['SeedDiff'].values
seed_spline_m = UnivariateSpline(
    sorted(seed_diff_m),
    feat_df_m.sort_values('SeedDiff')['Result'].values,
    s=len(seed_diff_m), ext=3,
)
print("Full dataset retraining complete.")

Full dataset retraining complete.


In [ ]:
def predict_submission_df(teams_df, season, all_feats, feat_cols, trained_models,
                          seeds_df, spline, blend_weight, massey_feats=None):
    """Generate all-matchup predictions for submission."""
    active = teams_df[teams_df.get('LastD1Season', pd.Series([season+1]*len(teams_df))) >= season]['TeamID'].tolist()
    rows = []
    for i, t1 in enumerate(active):
        for t2 in active[i+1:]:
            lo, hi = min(t1,t2), max(t1,t2)

            f1 = all_feats[(all_feats['Season']==season) & (all_feats['TeamID']==lo)]
            f2 = all_feats[(all_feats['Season']==season) & (all_feats['TeamID']==hi)]

            feat_row = {'SeedDiff': 0}
            for col in feat_cols:
                if col == 'SeedDiff': continue
                v1 = f1.iloc[0].get(col, 0) if len(f1) else 0
                v2 = f2.iloc[0].get(col, 0) if len(f2) else 0
                feat_row[f'{col}_1'] = v1
                feat_row[f'{col}_2'] = v2
                feat_row[f'{col}_Diff'] = v1 - v2

            def gs(tid):
                r = seeds_df[(seeds_df['Season']==season)&(seeds_df['TeamID']==tid)]
                return int(r.iloc[0]['Seed'][1:3]) if len(r) else 8
            feat_row['SeedDiff'] = gs(lo) - gs(hi)

            row_df = pd.DataFrame([feat_row])[model_feat_cols].fillna(0)
            preds = [clf.predict_proba(row_df)[0,1] for _, clf in trained_models]
            ensemble_pred = np.mean(preds)
            # spline(SeedDiff): SeedDiff<0 means lo has better seed -> higher win prob
            spline_pred = float(np.clip(spline(feat_row['SeedDiff']), 0.025, 0.975))
            final_pred = blend_weight * ensemble_pred + (1 - blend_weight) * spline_pred
            rows.append({'ID': f"{season}_{lo}_{hi}", 'Pred': np.clip(final_pred, 0.025, 0.975)})

    return pd.DataFrame(rows)

# Combine features for all seasons
all_feats_m = compact_m.copy()
all_feats_m = all_feats_m.merge(detailed_m, on=['Season','TeamID'], how='left')
all_feats_m = all_feats_m.merge(massey_multi_m, on=['Season','TeamID'], how='left')

sub_m = predict_submission_df(
    teams_m, CURRENT_SEASON, all_feats_m, model_feat_cols,
    trained_models_m, seeds_m, seed_spline_m, ENSEMBLE_WEIGHT,
)
print(f"Men's predictions: {len(sub_m)} matchups")

## Women's Ensemble (no Massey)

In [ ]:
SKIP_WOMENS = False
sub_w = pd.DataFrame(columns=['ID', 'Pred'])  # default empty if women's data unavailable

try:
    reg_w = pd.read_csv(f"{DATA_DIR}/WRegularSeasonCompactResults.csv")
    try:
        det_w = pd.read_csv(f"{DATA_DIR}/WRegularSeasonDetailedResults.csv")
    except FileNotFoundError:
        det_w = None
        print("No detailed results for women's — using compact only")
    tourney_w = pd.read_csv(f"{DATA_DIR}/WNCAATourneyCompactResults.csv")
    seeds_w = pd.read_csv(f"{DATA_DIR}/WNCAATourneySeeds.csv")
    teams_w = pd.read_csv(f"{DATA_DIR}/WTeams.csv")
    teams_m_ref = pd.read_csv(f"{DATA_DIR}/MTeams.csv")

    compact_w = compute_compact_features(reg_w)
    detailed_w = compute_detailed_features(det_w) if det_w is not None else None

    train_tourney_w = tourney_w[(tourney_w['Season'] >= 2010) & (tourney_w['Season'] < CURRENT_SEASON)]
    feat_df_w = build_matchup_df(train_tourney_w, compact_w, detailed_w, seeds_w, massey_feats=None)

    model_feat_cols_w = [c for c in feat_df_w.columns
                          if c not in ['Season','Team1ID','Team2ID','Result']]
    X_w = feat_df_w[model_feat_cols_w].fillna(0)
    y_w = feat_df_w['Result']
    groups_w = feat_df_w['Season']

    print(f"Women's training samples: {len(X_w)}, features: {len(model_feat_cols_w)}")

    # Train women's ensemble
    gkf_w = GroupKFold(n_splits=min(N_FOLDS, feat_df_w['Season'].nunique()))
    oof_preds_w = np.zeros((len(X_w), len(models)))

    for fold, (train_idx, val_idx) in enumerate(gkf_w.split(X_w, y_w, groups_w)):
        X_tr, X_val = X_w.iloc[train_idx], X_w.iloc[val_idx]
        y_tr, y_val = y_w.iloc[train_idx], y_w.iloc[val_idx]
        for mi, (name, clf) in enumerate(models):
            clf.fit(X_tr, y_tr)
            oof_preds_w[val_idx, mi] = clf.predict_proba(X_val)[:, 1]

    ensemble_oof_w = oof_preds_w.mean(axis=1)
    print(f"Women's OOF log loss: {log_loss(y_w, ensemble_oof_w):.4f}")

    trained_models_w = []
    for name, clf in models:
        clf.fit(X_w, y_w)
        trained_models_w.append((name, clf))

    seed_diff_w = feat_df_w['SeedDiff'].values
    seed_spline_w = UnivariateSpline(
        sorted(seed_diff_w),
        feat_df_w.sort_values('SeedDiff')['Result'].values,
        s=len(seed_diff_w), ext=3,
    )

    all_feats_w = compact_w.copy()
    if detailed_w is not None:
        all_feats_w = all_feats_w.merge(detailed_w, on=['Season','TeamID'], how='left')

    # Women's active teams via men's cross-reference
    active_names_w = set(teams_m_ref[teams_m_ref['LastD1Season'] >= CURRENT_SEASON]['TeamName'])
    teams_w_active = teams_w[teams_w['TeamName'].isin(active_names_w)].copy()
    teams_w_active['LastD1Season'] = CURRENT_SEASON + 1  # ensure included

    sub_w = predict_submission_df(
        teams_w_active, CURRENT_SEASON, all_feats_w, model_feat_cols_w,
        trained_models_w, seeds_w, seed_spline_w, ENSEMBLE_WEIGHT,
    )
    print(f"Women's predictions: {len(sub_w)} matchups")

except (FileNotFoundError, Exception) as e:
    print(f"Women's data not found in {DATA_DIR} — skipping women's predictions")
    print(f"  Reason: {e}")
    print(f"  Tip: Use DATA_DIR = '../data/2026' for full data including women's.")
    SKIP_WOMENS = True

In [ ]:
combined = pd.concat([sub_m, sub_w], ignore_index=True)
out_dir = f"../output/{CURRENT_SEASON}/xgb_ensemble"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/submission_combined.csv"
combined.to_csv(out_path, index=False)

print(f"Saved {len(combined)} rows to {out_path}")
print(f"  Men's:   {len(sub_m)}")
print(f"  Women's: {len(sub_w)}" + (" (SKIPPED — missing data)" if SKIP_WOMENS else ""))
combined.describe()